In [ ]:
# !rm -rf /kaggle/working/*

In [ ]:
!pip install wandb

# restart kernel

In [2]:
!pip install SimpleITK
!pip install monai==1.4.0
!pip install einops==0.8.1
!pip install progress_table
!pip install medpy
!pip install wandb

In [ ]:
# import os
# from pathlib import Path
# import SimpleITK as sitk

# BRATS_TRAIN_FOLDERS = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS2021_Training_Data"
# VALI_PATH = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS_2021_ValidationData"

# def explore_directory_structure(patient_dir, level=0):
#     """Recursively explore directory structure to find actual files."""
#     indent = "  " * level
#     print(f"{indent}📁 {patient_dir.name}/")
    
#     try:
#         for item in sorted(patient_dir.iterdir()):
#             if item.is_dir():
#                 explore_directory_structure(item, level + 1)
#             else:
#                 size = item.stat().st_size / 1024  # KB
#                 file_type = "NIfTI" if item.suffix in ['.nii', '.gz'] else "Other"
#                 print(f"{indent}  📄 {item.name} ({size:.1f} KB) - {file_type}")
#     except PermissionError:
#         print(f"{indent}  ⚠️ Permission denied")

# def find_actual_nifti_files(patient_dir):
#     """Find all actual NIfTI files (ignore 0-byte placeholders)."""
#     actual_files = []
    
#     # Search recursively for all .nii and .nii.gz files
#     for ext in ['*.nii', '*.nii.gz']:
#         for file_path in patient_dir.rglob(ext):
#             file_size = file_path.stat().st_size
#             # Only consider files > 1000 bytes (real data, not placeholders)
#             if file_size > 1000:
#                 actual_files.append(file_path)
    
#     return actual_files

# def diagnose_patient_complete(patient_dir):
#     """Complete diagnosis to find actual NIfTI files in nested directories."""
#     patient_id = patient_dir.name
#     print(f"\n{'='*70}")
#     print(f"🔍 DIAGNOSING PATIENT: {patient_id}")
#     print(f"{'='*70}")
    
#     # Step 1: Show directory structure
#     print("\n📂 1. DIRECTORY STRUCTURE:")
#     print("-" * 50)
#     explore_directory_structure(patient_dir)
    
#     # Step 2: Find actual NIfTI files
#     print("\n📁 2. ACTUAL NIfTI FILES (size > 1KB):")
#     print("-" * 50)
#     actual_files = find_actual_nifti_files(patient_dir)
    
#     if not actual_files:
#         print("   ❌ No valid NIfTI files found!")
#         return False
    
#     # Step 3: Map files to modalities (CRITICAL: t1ce must be checked BEFORE t1)
#     print("\n📊 3. MODALITY MAPPING:")
#     print("-" * 50)
    
#     # Order matters! Check more specific patterns first
#     modality_patterns = [
#         ('t1ce', ['t1ce', 't1_ce', 'T1CE', 'T1_CE']),
#         ('t1', ['t1', 'T1']),  # Check t1 AFTER t1ce to avoid false matches
#         ('t2', ['t2', 'T2']),
#         ('flair', ['flair', 'FLAIR']),
#         ('seg', ['seg', 'Seg', 'SEG', 'final_seg', 'seg_new'])
#     ]
    
#     # Initialize results
#     found_modalities = {modality: None for modality, _ in modality_patterns}
#     modality_shapes = {}
#     modality_sizes = {}
#     unknown_files = []
    
#     # Track which files have been assigned
#     assigned_files = set()
    
#     # First pass: assign files to modalities based on patterns
#     for file_path in actual_files:
#         file_name_lower = file_path.name.lower()
#         file_size = file_path.stat().st_size / (1024 * 1024)  # MB
#         assigned = False
        
#         # Check each modality pattern in priority order
#         for modality, patterns in modality_patterns:
#             if assigned:
#                 break
#             for pattern in patterns:
#                 if pattern.lower() in file_name_lower:
#                     # For t1, ensure it's not actually t1ce
#                     if modality == 't1' and ('ce' in file_name_lower or 't1ce' in file_name_lower):
#                         continue
                    
#                     # Check if this modality already found (prefer first match)
#                     if found_modalities[modality] is None:
#                         found_modalities[modality] = file_path
#                         modality_sizes[modality] = file_size
#                         assigned_files.add(file_path)
#                         assigned = True
                        
#                         # Try to read the file
#                         try:
#                             img = sitk.ReadImage(str(file_path))
#                             img_array = sitk.GetArrayFromImage(img)
#                             modality_shapes[modality] = img_array.shape
#                             status = "✅ VALID"
#                         except Exception as e:
#                             modality_shapes[modality] = f"ERROR: {str(e)[:50]}"
#                             status = "❌ CORRUPTED"
                        
#                         print(f"   {modality.upper():6s}: {file_path.name}")
#                         print(f"            Size: {file_size:.2f} MB | Status: {status}")
#                         print(f"            Shape: {modality_shapes[modality]}")
#                         break
#                     break
        
#         if not assigned:
#             unknown_files.append(file_path)
    
#     # Report any unknown files
#     for file_path in unknown_files:
#         file_size = file_path.stat().st_size / (1024 * 1024)
#         print(f"   UNKNOWN: {file_path.name} ({file_size:.2f} MB)")
    
#     # Step 4: Check completeness
#     print("\n✅ 4. VALIDATION SUMMARY:")
#     print("-" * 50)
    
#     all_present = True
#     missing_modalities = []
    
#     for modality, _ in modality_patterns:
#         if found_modalities[modality] is not None:
#             if modality in modality_shapes and modality_shapes[modality] is not None:
#                 print(f"   ✅ {modality.upper()}: FOUND - Shape: {modality_shapes[modality]}")
#             else:
#                 print(f"   ⚠️ {modality.upper()}: FOUND but CORRUPTED")
#                 all_present = False
#         else:
#             print(f"   ❌ {modality.upper()}: MISSING")
#             missing_modalities.append(modality)
#             all_present = False
    
#     # Check for shape consistency across modalities
#     shapes = [modality_shapes[m] for m in modality_shapes if m in modality_shapes and modality_shapes[m] is not None]
#     if len(shapes) > 1 and all(s == shapes[0] for s in shapes):
#         print(f"\n   📐 All modalities have consistent shape: {shapes[0]}")
#     elif len(shapes) > 1:
#         print(f"\n   ⚠️ WARNING: Inconsistent shapes detected!")
#         for modality in modality_shapes:
#             if modality in modality_shapes:
#                 print(f"      {modality}: {modality_shapes[modality]}")
    
#     if all_present and len(missing_modalities) == 0:
#         print(f"\n🎉 PATIENT {patient_id} IS COMPLETELY VALID!")
#         return True
#     else:
#         print(f"\n❌ PATIENT {patient_id} IS INVALID")
#         if missing_modalities:
#             print(f"   Missing modalities: {', '.join(missing_modalities)}")
#         return False

# def verify_dataset(dataset_path, dataset_name, max_patients=10):
#     """Verify multiple patients in a dataset."""
#     print(f"\n{'#'*70}")
#     print(f"🔍 VERIFYING DATASET: {dataset_name}")
#     print(f"   Path: {dataset_path}")
#     print(f"{'#'*70}")
    
#     dataset_path = Path(dataset_path)
#     if not dataset_path.exists():
#         print(f"❌ Dataset path does not exist: {dataset_path}")
#         return
    
#     patients = sorted([x for x in dataset_path.iterdir() if x.is_dir()])
#     print(f"\n📊 Total patient folders: {len(patients)}")
    
#     if not patients:
#         print("❌ No patient folders found!")
#         return
    
#     print(f"\n📋 Checking first {min(max_patients, len(patients))} patients...")
    
#     valid_patients = []
#     invalid_patients = []
#     shape_info = {}
    
#     for i, patient_dir in enumerate(patients[:max_patients]):
#         is_valid = diagnose_patient_complete(patient_dir)
#         if is_valid:
#             valid_patients.append(patient_dir.name)
#         else:
#             invalid_patients.append(patient_dir.name)
    
#     print(f"\n{'='*70}")
#     print(f"📊 SUMMARY for {dataset_name}:")
#     print(f"{'='*70}")
#     print(f"   Total patients: {len(patients)}")
#     print(f"   Checked: {len(valid_patients) + len(invalid_patients)}")
#     print(f"   ✅ Valid: {len(valid_patients)}")
#     print(f"   ❌ Invalid: {len(invalid_patients)}")
#     if len(valid_patients) + len(invalid_patients) > 0:
#         print(f"   Valid percentage: {len(valid_patients)/(len(valid_patients)+len(invalid_patients))*100:.1f}%")
    
#     if valid_patients:
#         print(f"\n   Valid patients: {', '.join(valid_patients[:5])}")
#         if len(valid_patients) > 5:
#             print(f"   ... and {len(valid_patients) - 5} more")
    
#     if invalid_patients:
#         print(f"\n   Invalid patients: {', '.join(invalid_patients[:5])}")
#         if len(invalid_patients) > 5:
#             print(f"   ... and {len(invalid_patients) - 5} more")
    
#     print(f"{'='*70}")
    
#     return valid_patients, invalid_patients

# def quick_check_patient(patient_id, dataset_path=BRATS_TRAIN_FOLDERS):
#     """Quick check a specific patient by ID."""
#     patient_dir = Path(dataset_path) / patient_id
#     if not patient_dir.exists():
#         print(f"❌ Patient {patient_id} not found in {dataset_path}")
#         return False
#     return diagnose_patient_complete(patient_dir)

# # Run verification
# if __name__ == "__main__":
#     print("🚀 STARTING DATASET VERIFICATION")
#     print("="*70)
    
#     # Verify training dataset
#     train_valid, train_invalid = verify_dataset(BRATS_TRAIN_FOLDERS, "TRAINING DATASET", max_patients=10)
    
#     # Verify validation dataset
#     val_valid, val_invalid = verify_dataset(VALI_PATH, "VALIDATION DATASET", max_patients=10)
    
#     # Optionally check a specific patient
#     print("\n" + "="*70)
#     print("🔍 CHECKING SPECIFIC PATIENT: BraTS2021_00000")
#     print("="*70)
#     quick_check_patient("BraTS2021_00000", BRATS_TRAIN_FOLDERS)
    
#     print("\n✨ Verification complete!")
#     print(f"\n📈 Overall: Training dataset has {len(train_valid)} valid out of 10 checked ({len(train_valid)/10*100:.0f}%)")
#     print(f"           Validation dataset has {len(val_valid)} valid out of 10 checked ({len(val_valid)/10*100:.0f}%)")

## Training

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import SimpleITK as sitk
from pathlib import Path
from tqdm import tqdm

# MONAI components
from monai.losses import DiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from medpy.metric import binary

# ======================== CONFIGURATION ========================

BRATS_TRAIN_FOLDERS = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS2021_Training_Data"
VALI_PATH = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS_2021_ValidationData"


# ======================== FIXED FILE FINDER WITH CORRECT PATTERN MATCHING ========================

def find_actual_nifti_file(patient_dir, modality):
    """
    Find the actual NIfTI file by looking inside subdirectories.
    Priority order for matching to avoid t1ce being matched as t1.
    """
    # Define search patterns with priority (more specific first)
    pattern_map = {
        't1ce': ['t1ce', 't1_ce', 'T1CE', 'T1_CE'],
        't1': ['t1', 'T1'],
        't2': ['t2', 'T2'],
        'flair': ['flair', 'FLAIR'],
        'seg': ['seg', 'Seg', 'SEG', 'final_seg', 'seg_new']
    }
    
    # First, try to find in subdirectories (the placeholder directories)
    placeholder_dir = patient_dir / f"{patient_dir.name}_{modality}.nii"
    if placeholder_dir.exists() and placeholder_dir.is_dir():
        # Look for actual .nii files inside
        for file_path in placeholder_dir.iterdir():
            if file_path.suffix == '.nii' and file_path.stat().st_size > 1000:
                # Verify it's the correct modality
                file_name_lower = file_path.name.lower()
                for pattern in pattern_map[modality]:
                    if pattern.lower() in file_name_lower:
                        return file_path
    
    # If not found, search recursively
    for file_path in patient_dir.rglob("*.nii"):
        if file_path.stat().st_size < 1000:
            continue
        
        file_name_lower = file_path.name.lower()
        
        # For t1ce, ensure we don't match it as t1
        if modality == 't1ce':
            if 't1ce' in file_name_lower or 't1_ce' in file_name_lower:
                return file_path
        elif modality == 't1':
            # Match t1 but NOT t1ce
            if 't1' in file_name_lower and 'ce' not in file_name_lower and 't1ce' not in file_name_lower:
                return file_path
        else:
            for pattern in pattern_map[modality]:
                if pattern.lower() in file_name_lower:
                    return file_path
    
    return None


def is_valid_nifti(file_path):
    """Check if file is a valid NIfTI file."""
    if file_path is None:
        return False
    if not file_path.exists():
        return False
    if not file_path.is_file():
        return False
    if file_path.stat().st_size < 1000:  # Real NIfTI files are > 1KB
        return False
    try:
        img = sitk.ReadImage(str(file_path))
        img_array = sitk.GetArrayFromImage(img)
        return img_array is not None and img_array.size > 0
    except Exception as e:
        return False


class BratsDataset(Dataset):
    def __init__(self, patients_dir, training=True, normalisation="zscore"):
        self.patients_dir = patients_dir
        self.training = training
        self.normalisation = normalisation
        
        self.patients = []
        self.invalid_patients = []
        self.patient_files = {}  # Store file paths for each patient
        
        all_patients = sorted([x for x in Path(patients_dir).iterdir() if x.is_dir()])
        
        print(f"Validating {len(all_patients)} patients...")
        for patient_dir in all_patients:
            is_valid, files = self._validate_patient(patient_dir)
            if is_valid:
                self.patients.append(patient_dir)
                self.patient_files[patient_dir.name] = files
            else:
                self.invalid_patients.append(patient_dir.name)
        
        print(f"Found {len(self.patients)} valid patients, {len(self.invalid_patients)} invalid")
        if self.invalid_patients:
            print(f"Invalid patients (first 10): {self.invalid_patients[:10]}")
        
    def _validate_patient(self, patient_dir):
        """Check if patient has all required files and return file paths."""
        modalities = ['t1', 't1ce', 't2', 'flair', 'seg']
        files_found = {}
        
        for modality in modalities:
            file_path = find_actual_nifti_file(patient_dir, modality)
            if is_valid_nifti(file_path):
                files_found[modality] = file_path
            else:
                return False, None
        
        return True, files_found
    
    def load_nii_safe(self, path):
        """Safely load NIfTI file."""
        try:
            img = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
            if img is None or img.size == 0:
                raise ValueError(f"Empty image: {path}")
            return img
        except Exception as e:
            print(f"Error loading {path}: {e}")
            raise
    
    def __len__(self):
        return len(self.patients)
    
    def __getitem__(self, idx):
        patient_dir = self.patients[idx]
        patient_id = patient_dir.name
        files = self.patient_files[patient_id]
        
        # Load all modalities
        patient_image = {}
        for modality in ['t1', 't1ce', 't2', 'flair']:
            patient_image[modality] = self.load_nii_safe(files[modality])
        
        # Load segmentation
        patient_label = self.load_nii_safe(files['seg'])
        
        # Convert to ET, TC, WT
        et = (patient_label == 4).astype(np.float32)
        tc = np.logical_or(patient_label == 4, patient_label == 1).astype(np.float32)
        wt = np.logical_or(tc, patient_label == 2).astype(np.float32)
        patient_label = np.stack([et, tc, wt], axis=0)
        
        # Normalisation
        for key in patient_image:
            if self.normalisation == "minmax":
                patient_image[key] = self._irm_min_max_preprocess(patient_image[key])
            elif self.normalisation == "zscore":
                patient_image[key] = self._zscore_normalise(patient_image[key])
        
        # Stack modalities
        patient_image = np.stack([patient_image['t1'], patient_image['t1ce'], 
                                   patient_image['t2'], patient_image['flair']], axis=0)
        
        # Crop to non-zero region
        z_indexes, y_indexes, x_indexes = np.nonzero(np.sum(patient_image, axis=0) != 0)
        if len(z_indexes) > 0:
            zmin, ymin, xmin = [max(0, int(np.min(arr)) - 1) for arr in (z_indexes, y_indexes, x_indexes)]
            zmax, ymax, xmax = [int(np.max(arr)) + 1 for arr in (z_indexes, y_indexes, x_indexes)]
            patient_image = patient_image[:, zmin:zmax, ymin:ymax, xmin:xmax]
            patient_label = patient_label[:, zmin:zmax, ymin:ymax, xmin:xmax]
        
        if self.training:
            patient_image, patient_label = self._pad_or_crop_image(patient_image, patient_label, target_size=(128, 144, 144))
            patient_image = self._resize_volume(patient_image, (128, 128, 128))
            patient_label = self._resize_volume(patient_label, (128, 128, 128), is_label=True)
            
            # Data augmentation
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=1)
                patient_label = np.flip(patient_label, axis=1)
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=2)
                patient_label = np.flip(patient_label, axis=2)
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=3)
                patient_label = np.flip(patient_label, axis=3)
            if random.random() < 0.5:
                scale = 0.8 + 0.4 * random.random()
                patient_image = patient_image * scale
            if random.random() < 0.5:
                shift = 0.2 * (random.random() - 0.5)
                patient_image = patient_image + shift
        else:
            patient_image, patient_label = self._pad_or_crop_image_label(patient_image, patient_label, target_size=(128, 128, 128))
        
        return torch.from_numpy(patient_image.astype(np.float32)), torch.from_numpy(patient_label.astype(np.float32))
    
    def _zscore_normalise(self, img):
        img = img.astype(np.float32)
        nonzero = img[img > 0]
        if len(nonzero) > 0:
            mean, std = nonzero.mean(), nonzero.std()
            if std > 0:
                img = (img - mean) / std
        return img
    
    def _irm_min_max_preprocess(self, image, low_perc=1, high_perc=99):
        non_zeros = image > 0
        if np.sum(non_zeros) == 0:
            return image
        low, high = np.percentile(image[non_zeros], [low_perc, high_perc])
        image = np.clip(image, low, high)
        min_, max_ = np.min(image), np.max(image)
        scale = max_ - min_
        if scale > 0:
            image = (image - min_) / scale
        return image
    
    def _get_crop_slice(self, target_size, dim):
        if dim > target_size:
            crop_extent = dim - target_size
            left = random.randint(0, crop_extent)
            right = crop_extent - left
            return slice(left, dim - right)
        elif dim <= target_size:
            return slice(0, dim)
    
    def _get_crop_slice2(self, target_size, dim):
        if dim > target_size:
            start = (dim - target_size) // 2
            end = start + target_size
            return slice(start, end)
        elif dim <= target_size:
            return slice(0, dim)
    
    def _get_left_right_idx_should_pad(self, target_size, dim):
        if dim >= target_size:
            return [False]
        elif dim < target_size:
            pad_extent = target_size - dim
            left = random.randint(0, pad_extent)
            right = pad_extent - left
            return True, left, right
    
    def _pad_or_crop_image(self, image, seg=None, target_size=(128, 144, 144)):
        c, z, y, x = image.shape
        z_slice, y_slice, x_slice = [self._get_crop_slice(target, dim) for target, dim in zip(target_size, (z, y, x))]
        image = image[:, z_slice, y_slice, x_slice]
        if seg is not None:
            seg = seg[:, z_slice, y_slice, x_slice]
        todos = [self._get_left_right_idx_should_pad(size, dim) for size, dim in zip(target_size, [z, y, x])]
        padlist = [(0, 0)]
        for to_pad in todos:
            if to_pad[0]:
                padlist.append((to_pad[1], to_pad[2]))
            else:
                padlist.append((0, 0))
        image = np.pad(image, padlist)
        if seg is not None:
            seg = np.pad(seg, padlist)
            return image, seg
        return image
    
    def _pad_or_crop_image_label(self, image, seg=None, target_size=(128, 128, 128)):
        c, z, y, x = image.shape
        z_slice, y_slice, x_slice = [self._get_crop_slice2(target, dim) for target, dim in zip(target_size, (z, y, x))]
        image = image[:, z_slice, y_slice, x_slice]
        if seg is not None:
            seg = seg[:, z_slice, y_slice, x_slice]
        if seg is not None:
            return image, seg
        return image
    
    def _resize_volume(self, volume, target_size, is_label=False):
        import torch.nn.functional as F_torch
        tensor = torch.from_numpy(volume).unsqueeze(0)
        mode = 'nearest' if is_label else 'trilinear'
        align = False if not is_label else None
        resized = F_torch.interpolate(tensor, size=target_size, mode=mode, align_corners=align)
        return resized.squeeze(0).numpy()


# ======================== MODEL ARCHITECTURE (same as before) ========================

class EfficientCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)
        self.to_out = nn.Linear(dim, dim)
        
    def forward(self, a, b):
        B, C, D, H, W = a.shape
        a_seq = a.flatten(2).transpose(1, 2)
        b_seq = b.flatten(2).transpose(1, 2)
        
        q = self.to_q(a_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        k = self.to_k(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        v = self.to_v(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, -1, C)
        out = self.to_out(out)
        out = a_seq + out
        out = out.transpose(1, 2).view(B, C, D, H, W)
        return out


class LightweightDenoiser(nn.Module):
    def __init__(self, in_channels=4, base_filters=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*4),
            nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*4, base_filters*2, 3, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*2, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, in_channels, 3, padding=1),
        )
        
    def forward(self, x):
        features = self.encoder(x)
        recon = self.decoder(features)
        return torch.sigmoid(recon), features


class CMADS_Net_MemoryEfficient(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_filters=16):
        super().__init__()
        
        def make_encoder():
            return nn.ModuleList([
                nn.Sequential(
                    nn.Conv3d(in_channels, base_filters, 3, padding=1),
                    nn.InstanceNorm3d(base_filters),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*2),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*4),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*4, base_filters*8, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*8),
                    nn.LeakyReLU(),
                )
            ])
        
        self.encoder_clean = make_encoder()
        self.encoder_noisy = make_encoder()
        self.cross_attn = EfficientCrossAttention(base_filters*8, num_heads=4)
        self.denoiser = LightweightDenoiser(in_channels, base_filters)
        
        self.decoder = nn.ModuleList([
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*8 + base_filters*4, base_filters*4, 3, padding=1),
                nn.InstanceNorm3d(base_filters*4),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*4 + base_filters*2, base_filters*2, 3, padding=1),
                nn.InstanceNorm3d(base_filters*2),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*2 + base_filters, base_filters, 3, padding=1),
                nn.InstanceNorm3d(base_filters),
                nn.LeakyReLU(),
            ),
            nn.Conv3d(base_filters, out_channels, 1)
        ])
        
    def forward(self, x_clean, noise_std=0.0, return_recon=False):
        if self.training and noise_std > 0:
            noise = torch.randn_like(x_clean) * noise_std
            x_noisy = x_clean + noise
        else:
            x_noisy = x_clean
        
        recon, _ = self.denoiser(x_noisy)
        
        clean_feats = []
        y = x_clean
        for block in self.encoder_clean:
            y = block(y)
            clean_feats.append(y)
        
        noisy_feats = []
        z = x_noisy
        for block in self.encoder_noisy:
            z = block(z)
            noisy_feats.append(z)
        
        fused_bottleneck = self.cross_attn(clean_feats[3], noisy_feats[3])
        
        recon_error_full = torch.abs(recon - x_clean).mean(dim=1, keepdim=True)
        
        error_map_size1 = F.interpolate(recon_error_full, size=clean_feats[1].shape[2:], mode='trilinear', align_corners=False)
        error_map_size2 = F.interpolate(recon_error_full, size=clean_feats[2].shape[2:], mode='trilinear', align_corners=False)
        
        calibrated_skip2 = clean_feats[2] * (1 + error_map_size2)
        calibrated_skip1 = clean_feats[1] * (1 + error_map_size1)
        calibrated_skip0 = clean_feats[0] * (1 + recon_error_full)
        
        d3 = self.decoder[0][0](fused_bottleneck)
        d3 = torch.cat([d3, calibrated_skip2], dim=1)
        d3 = self.decoder[0][1:](d3)
        
        d2 = self.decoder[1][0](d3)
        d2 = torch.cat([d2, calibrated_skip1], dim=1)
        d2 = self.decoder[1][1:](d2)
        
        d1 = self.decoder[2][0](d2)
        d1 = torch.cat([d1, calibrated_skip0], dim=1)
        d1 = self.decoder[2][1:](d1)
        
        logits = self.decoder[3](d1)
        
        if return_recon:
            return logits, recon
        return logits


# ======================== LOSS AND TRAINING ========================

class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=1.0, recon_weight=0.05):
        super().__init__()
        self.dice_loss = DiceLoss(sigmoid=True, softmax=False, to_onehot_y=False)
        self.dice_weight = dice_weight
        self.recon_weight = recon_weight
    
    def forward(self, seg_logits, seg_target, recon_image, orig_image):
        seg_loss = self.dice_loss(seg_logits, seg_target)
        recon_loss = F.l1_loss(recon_image, orig_image)
        total = self.dice_weight * seg_loss + self.recon_weight * recon_loss
        return total, seg_loss, recon_loss


def train_epoch(model, loader, optimizer, loss_fn, epoch, max_epochs, device):
    model.train()
    total_loss = 0
    noise_std = min(0.15, 0.03 + 0.12 * epoch / max_epochs)
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{max_epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        seg_logits, recon = model(images, noise_std=noise_std, return_recon=True)
        loss, seg_l, recon_l = loss_fn(seg_logits, labels, recon, images)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item(), seg=seg_l.item(), recon=recon_l.item())
    
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch', get_not_nans=True)
    all_hd95 = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = sliding_window_inference(images, (128,128,128), 1, model, overlap=0.5)
            preds = torch.sigmoid(logits) > 0.5
            
            pred_list = [p for p in preds]
            label_list = [l for l in labels]
            dice_metric(y_pred=pred_list, y=label_list)
            
            for i in range(preds.shape[0]):
                for j in range(preds.shape[1]):
                    p = preds[i, j].cpu().numpy()
                    t = labels[i, j].cpu().numpy()
                    
                    if t.sum() == 0:
                        if p.sum() == 0:
                            hd95 = 0.0
                        else:
                            hd95 = 50.0
                    elif p.sum() == 0:
                        hd95 = 50.0
                    else:
                        try:
                            hd95 = binary.hd95(p, t, voxelspacing=None)
                            if np.isnan(hd95) or np.isinf(hd95):
                                hd95 = 50.0
                        except Exception:
                            hd95 = 50.0
                    
                    all_hd95.append(hd95)
    
    dice = dice_metric.aggregate()[0].cpu().numpy()
    mean_hd95 = np.mean(all_hd95) if all_hd95 else 50.0
    return dice, mean_hd95


def main():
    set_determinism(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Data
    train_dataset = BratsDataset(BRATS_TRAIN_FOLDERS, training=True, normalisation="zscore")
    val_dataset = BratsDataset(VALI_PATH, training=False, normalisation="zscore")
    
    if len(train_dataset) == 0:
        print("ERROR: No valid training patients found!")
        return
    
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Model
    model = CMADS_Net_MemoryEfficient(in_channels=4, out_channels=3, base_filters=16).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Optimizer
    optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=100)
    loss_fn = CombinedLoss(dice_weight=1.0, recon_weight=0.05)
    
    # Training
    best_dice = 0.0
    for epoch in range(1, 101):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, epoch, 100, device)
        scheduler.step()
        
        if epoch % 5 == 0 or epoch == 100:
            dice_scores, hd95 = validate(model, val_loader, device)
            avg_dice = np.mean(dice_scores)
            print(f"Epoch {epoch:3d} | Loss: {train_loss:.4f} | Dice ET/TC/WT: {dice_scores[0]:.4f}/{dice_scores[1]:.4f}/{dice_scores[2]:.4f} (Avg: {avg_dice:.4f}) | HD95: {hd95:.2f}")
            
            if avg_dice > best_dice:
                best_dice = avg_dice
                torch.save(model.state_dict(), "best_cmads_model.pth")
                print(f"*** New best: {best_dice:.4f} ***")
    
    print(f"\nTraining complete. Best Dice: {best_dice:.4f}")


if __name__ == "__main__":
    main()

## Resume Training

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import SimpleITK as sitk
from pathlib import Path
from tqdm import tqdm

# MONAI components
from monai.losses import DiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from medpy.metric import binary

# ======================== CONFIGURATION ========================

BRATS_TRAIN_FOLDERS = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS2021_Training_Data"
VALI_PATH = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS_2021_ValidationData"


# ======================== FIXED FILE FINDER WITH CORRECT PATTERN MATCHING ========================

def find_actual_nifti_file(patient_dir, modality):
    """
    Find the actual NIfTI file by looking inside subdirectories.
    Priority order for matching to avoid t1ce being matched as t1.
    """
    # Define search patterns with priority (more specific first)
    pattern_map = {
        't1ce': ['t1ce', 't1_ce', 'T1CE', 'T1_CE'],
        't1': ['t1', 'T1'],
        't2': ['t2', 'T2'],
        'flair': ['flair', 'FLAIR'],
        'seg': ['seg', 'Seg', 'SEG', 'final_seg', 'seg_new']
    }
    
    # First, try to find in subdirectories (the placeholder directories)
    placeholder_dir = patient_dir / f"{patient_dir.name}_{modality}.nii"
    if placeholder_dir.exists() and placeholder_dir.is_dir():
        # Look for actual .nii files inside
        for file_path in placeholder_dir.iterdir():
            if file_path.suffix == '.nii' and file_path.stat().st_size > 1000:
                # Verify it's the correct modality
                file_name_lower = file_path.name.lower()
                for pattern in pattern_map[modality]:
                    if pattern.lower() in file_name_lower:
                        return file_path
    
    # If not found, search recursively
    for file_path in patient_dir.rglob("*.nii"):
        if file_path.stat().st_size < 1000:
            continue
        
        file_name_lower = file_path.name.lower()
        
        # For t1ce, ensure we don't match it as t1
        if modality == 't1ce':
            if 't1ce' in file_name_lower or 't1_ce' in file_name_lower:
                return file_path
        elif modality == 't1':
            # Match t1 but NOT t1ce
            if 't1' in file_name_lower and 'ce' not in file_name_lower and 't1ce' not in file_name_lower:
                return file_path
        else:
            for pattern in pattern_map[modality]:
                if pattern.lower() in file_name_lower:
                    return file_path
    
    return None


def is_valid_nifti(file_path):
    """Check if file is a valid NIfTI file."""
    if file_path is None:
        return False
    if not file_path.exists():
        return False
    if not file_path.is_file():
        return False
    if file_path.stat().st_size < 1000:  # Real NIfTI files are > 1KB
        return False
    try:
        img = sitk.ReadImage(str(file_path))
        img_array = sitk.GetArrayFromImage(img)
        return img_array is not None and img_array.size > 0
    except Exception as e:
        return False


class BratsDataset(Dataset):
    def __init__(self, patients_dir, training=True, normalisation="zscore"):
        self.patients_dir = patients_dir
        self.training = training
        self.normalisation = normalisation
        
        self.patients = []
        self.invalid_patients = []
        self.patient_files = {}  # Store file paths for each patient
        
        all_patients = sorted([x for x in Path(patients_dir).iterdir() if x.is_dir()])
        
        print(f"Validating {len(all_patients)} patients...")
        for patient_dir in all_patients:
            is_valid, files = self._validate_patient(patient_dir)
            if is_valid:
                self.patients.append(patient_dir)
                self.patient_files[patient_dir.name] = files
            else:
                self.invalid_patients.append(patient_dir.name)
        
        print(f"Found {len(self.patients)} valid patients, {len(self.invalid_patients)} invalid")
        if self.invalid_patients:
            print(f"Invalid patients (first 10): {self.invalid_patients[:10]}")
        
    def _validate_patient(self, patient_dir):
        """Check if patient has all required files and return file paths."""
        modalities = ['t1', 't1ce', 't2', 'flair', 'seg']
        files_found = {}
        
        for modality in modalities:
            file_path = find_actual_nifti_file(patient_dir, modality)
            if is_valid_nifti(file_path):
                files_found[modality] = file_path
            else:
                return False, None
        
        return True, files_found
    
    def load_nii_safe(self, path):
        """Safely load NIfTI file."""
        try:
            img = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
            if img is None or img.size == 0:
                raise ValueError(f"Empty image: {path}")
            return img
        except Exception as e:
            print(f"Error loading {path}: {e}")
            raise
    
    def __len__(self):
        return len(self.patients)
    
    def __getitem__(self, idx):
        patient_dir = self.patients[idx]
        patient_id = patient_dir.name
        files = self.patient_files[patient_id]
        
        # Load all modalities
        patient_image = {}
        for modality in ['t1', 't1ce', 't2', 'flair']:
            patient_image[modality] = self.load_nii_safe(files[modality])
        
        # Load segmentation
        patient_label = self.load_nii_safe(files['seg'])
        
        # Convert to ET, TC, WT
        et = (patient_label == 4).astype(np.float32)
        tc = np.logical_or(patient_label == 4, patient_label == 1).astype(np.float32)
        wt = np.logical_or(tc, patient_label == 2).astype(np.float32)
        patient_label = np.stack([et, tc, wt], axis=0)
        
        # Normalisation
        for key in patient_image:
            if self.normalisation == "minmax":
                patient_image[key] = self._irm_min_max_preprocess(patient_image[key])
            elif self.normalisation == "zscore":
                patient_image[key] = self._zscore_normalise(patient_image[key])
        
        # Stack modalities
        patient_image = np.stack([patient_image['t1'], patient_image['t1ce'], 
                                   patient_image['t2'], patient_image['flair']], axis=0)
        
        # Crop to non-zero region
        z_indexes, y_indexes, x_indexes = np.nonzero(np.sum(patient_image, axis=0) != 0)
        if len(z_indexes) > 0:
            zmin, ymin, xmin = [max(0, int(np.min(arr)) - 1) for arr in (z_indexes, y_indexes, x_indexes)]
            zmax, ymax, xmax = [int(np.max(arr)) + 1 for arr in (z_indexes, y_indexes, x_indexes)]
            patient_image = patient_image[:, zmin:zmax, ymin:ymax, xmin:xmax]
            patient_label = patient_label[:, zmin:zmax, ymin:ymax, xmin:xmax]
        
        if self.training:
            patient_image, patient_label = self._pad_or_crop_image(patient_image, patient_label, target_size=(128, 144, 144))
            patient_image = self._resize_volume(patient_image, (128, 128, 128))
            patient_label = self._resize_volume(patient_label, (128, 128, 128), is_label=True)
            
            # Data augmentation
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=1)
                patient_label = np.flip(patient_label, axis=1)
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=2)
                patient_label = np.flip(patient_label, axis=2)
            if random.random() < 0.5:
                patient_image = np.flip(patient_image, axis=3)
                patient_label = np.flip(patient_label, axis=3)
            if random.random() < 0.5:
                scale = 0.8 + 0.4 * random.random()
                patient_image = patient_image * scale
            if random.random() < 0.5:
                shift = 0.2 * (random.random() - 0.5)
                patient_image = patient_image + shift
        else:
            patient_image, patient_label = self._pad_or_crop_image_label(patient_image, patient_label, target_size=(128, 128, 128))
        
        return torch.from_numpy(patient_image.astype(np.float32)), torch.from_numpy(patient_label.astype(np.float32))
    
    def _zscore_normalise(self, img):
        img = img.astype(np.float32)
        nonzero = img[img > 0]
        if len(nonzero) > 0:
            mean, std = nonzero.mean(), nonzero.std()
            if std > 0:
                img = (img - mean) / std
        return img
    
    def _irm_min_max_preprocess(self, image, low_perc=1, high_perc=99):
        non_zeros = image > 0
        if np.sum(non_zeros) == 0:
            return image
        low, high = np.percentile(image[non_zeros], [low_perc, high_perc])
        image = np.clip(image, low, high)
        min_, max_ = np.min(image), np.max(image)
        scale = max_ - min_
        if scale > 0:
            image = (image - min_) / scale
        return image
    
    def _get_crop_slice(self, target_size, dim):
        if dim > target_size:
            crop_extent = dim - target_size
            left = random.randint(0, crop_extent)
            right = crop_extent - left
            return slice(left, dim - right)
        elif dim <= target_size:
            return slice(0, dim)
    
    def _get_crop_slice2(self, target_size, dim):
        if dim > target_size:
            start = (dim - target_size) // 2
            end = start + target_size
            return slice(start, end)
        elif dim <= target_size:
            return slice(0, dim)
    
    def _get_left_right_idx_should_pad(self, target_size, dim):
        if dim >= target_size:
            return [False]
        elif dim < target_size:
            pad_extent = target_size - dim
            left = random.randint(0, pad_extent)
            right = pad_extent - left
            return True, left, right
    
    def _pad_or_crop_image(self, image, seg=None, target_size=(128, 144, 144)):
        c, z, y, x = image.shape
        z_slice, y_slice, x_slice = [self._get_crop_slice(target, dim) for target, dim in zip(target_size, (z, y, x))]
        image = image[:, z_slice, y_slice, x_slice]
        if seg is not None:
            seg = seg[:, z_slice, y_slice, x_slice]
        todos = [self._get_left_right_idx_should_pad(size, dim) for size, dim in zip(target_size, [z, y, x])]
        padlist = [(0, 0)]
        for to_pad in todos:
            if to_pad[0]:
                padlist.append((to_pad[1], to_pad[2]))
            else:
                padlist.append((0, 0))
        image = np.pad(image, padlist)
        if seg is not None:
            seg = np.pad(seg, padlist)
            return image, seg
        return image
    
    def _pad_or_crop_image_label(self, image, seg=None, target_size=(128, 128, 128)):
        c, z, y, x = image.shape
        z_slice, y_slice, x_slice = [self._get_crop_slice2(target, dim) for target, dim in zip(target_size, (z, y, x))]
        image = image[:, z_slice, y_slice, x_slice]
        if seg is not None:
            seg = seg[:, z_slice, y_slice, x_slice]
        if seg is not None:
            return image, seg
        return image
    
    def _resize_volume(self, volume, target_size, is_label=False):
        import torch.nn.functional as F_torch
        tensor = torch.from_numpy(volume).unsqueeze(0)
        mode = 'nearest' if is_label else 'trilinear'
        align = False if not is_label else None
        resized = F_torch.interpolate(tensor, size=target_size, mode=mode, align_corners=align)
        return resized.squeeze(0).numpy()


# ======================== MODEL ARCHITECTURE (same as before) ========================

class EfficientCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)
        self.to_out = nn.Linear(dim, dim)
        
    def forward(self, a, b):
        B, C, D, H, W = a.shape
        a_seq = a.flatten(2).transpose(1, 2)
        b_seq = b.flatten(2).transpose(1, 2)
        
        q = self.to_q(a_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        k = self.to_k(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        v = self.to_v(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, -1, C)
        out = self.to_out(out)
        out = a_seq + out
        out = out.transpose(1, 2).view(B, C, D, H, W)
        return out


class LightweightDenoiser(nn.Module):
    def __init__(self, in_channels=4, base_filters=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*4),
            nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*4, base_filters*2, 3, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*2, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, in_channels, 3, padding=1),
        )
        
    def forward(self, x):
        features = self.encoder(x)
        recon = self.decoder(features)
        return torch.sigmoid(recon), features


class CMADS_Net_MemoryEfficient(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_filters=16):
        super().__init__()
        
        def make_encoder():
            return nn.ModuleList([
                nn.Sequential(
                    nn.Conv3d(in_channels, base_filters, 3, padding=1),
                    nn.InstanceNorm3d(base_filters),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*2),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*4),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*4, base_filters*8, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*8),
                    nn.LeakyReLU(),
                )
            ])
        
        self.encoder_clean = make_encoder()
        self.encoder_noisy = make_encoder()
        self.cross_attn = EfficientCrossAttention(base_filters*8, num_heads=4)
        self.denoiser = LightweightDenoiser(in_channels, base_filters)
        
        self.decoder = nn.ModuleList([
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*8 + base_filters*4, base_filters*4, 3, padding=1),
                nn.InstanceNorm3d(base_filters*4),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*4 + base_filters*2, base_filters*2, 3, padding=1),
                nn.InstanceNorm3d(base_filters*2),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*2 + base_filters, base_filters, 3, padding=1),
                nn.InstanceNorm3d(base_filters),
                nn.LeakyReLU(),
            ),
            nn.Conv3d(base_filters, out_channels, 1)
        ])
        
    def forward(self, x_clean, noise_std=0.0, return_recon=False):
        if self.training and noise_std > 0:
            noise = torch.randn_like(x_clean) * noise_std
            x_noisy = x_clean + noise
        else:
            x_noisy = x_clean
        
        recon, _ = self.denoiser(x_noisy)
        
        clean_feats = []
        y = x_clean
        for block in self.encoder_clean:
            y = block(y)
            clean_feats.append(y)
        
        noisy_feats = []
        z = x_noisy
        for block in self.encoder_noisy:
            z = block(z)
            noisy_feats.append(z)
        
        fused_bottleneck = self.cross_attn(clean_feats[3], noisy_feats[3])
        
        recon_error_full = torch.abs(recon - x_clean).mean(dim=1, keepdim=True)
        
        error_map_size1 = F.interpolate(recon_error_full, size=clean_feats[1].shape[2:], mode='trilinear', align_corners=False)
        error_map_size2 = F.interpolate(recon_error_full, size=clean_feats[2].shape[2:], mode='trilinear', align_corners=False)
        
        calibrated_skip2 = clean_feats[2] * (1 + error_map_size2)
        calibrated_skip1 = clean_feats[1] * (1 + error_map_size1)
        calibrated_skip0 = clean_feats[0] * (1 + recon_error_full)
        
        d3 = self.decoder[0][0](fused_bottleneck)
        d3 = torch.cat([d3, calibrated_skip2], dim=1)
        d3 = self.decoder[0][1:](d3)
        
        d2 = self.decoder[1][0](d3)
        d2 = torch.cat([d2, calibrated_skip1], dim=1)
        d2 = self.decoder[1][1:](d2)
        
        d1 = self.decoder[2][0](d2)
        d1 = torch.cat([d1, calibrated_skip0], dim=1)
        d1 = self.decoder[2][1:](d1)
        
        logits = self.decoder[3](d1)
        
        if return_recon:
            return logits, recon
        return logits


# ======================== LOSS AND TRAINING ========================

class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=1.0, recon_weight=0.05):
        super().__init__()
        self.dice_loss = DiceLoss(sigmoid=True, softmax=False, to_onehot_y=False)
        self.dice_weight = dice_weight
        self.recon_weight = recon_weight
    
    def forward(self, seg_logits, seg_target, recon_image, orig_image):
        seg_loss = self.dice_loss(seg_logits, seg_target)
        recon_loss = F.l1_loss(recon_image, orig_image)
        total = self.dice_weight * seg_loss + self.recon_weight * recon_loss
        return total, seg_loss, recon_loss

import glob

def save_checkpoint(state, filename="checkpoint.pth.tar"):
    """Save checkpoint with all training state."""
    torch.save(state, filename)
    print(f"Checkpoint saved to {filename}")

def load_checkpoint(model, optimizer, scheduler, filename="checkpoint.pth.tar", device='cuda'):
    """Load checkpoint and return (start_epoch, best_dice)."""
    if os.path.isfile(filename):
        print(f"Loading checkpoint '{filename}'")
        # checkpoint = torch.load(filename, map_location=device)
        checkpoint = torch.load(filename, map_location=device, weights_only=False)
        start_epoch = checkpoint['epoch'] + 1  
        best_dice = checkpoint['best_dice']
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(f"Loaded checkpoint (epoch {checkpoint['epoch']}, best Dice {best_dice:.4f})")
        return start_epoch, best_dice
    else:
        print(f"No checkpoint found at '{filename}', starting from scratch.")
        return 1, 0.0

def save_latest_checkpoint(model, optimizer, scheduler, epoch, best_dice, loss_avg=None, filename="latest_checkpoint.pth.tar"):
    """Save current training state."""
    state = {
        'epoch': epoch,
        'best_dice': best_dice,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }
    if loss_avg is not None:
        state['train_loss'] = loss_avg
    save_checkpoint(state, filename)

def train_epoch(model, loader, optimizer, loss_fn, epoch, max_epochs, device):
    model.train()
    total_loss = 0
    noise_std = min(0.15, 0.03 + 0.12 * epoch / max_epochs)
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{max_epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        seg_logits, recon = model(images, noise_std=noise_std, return_recon=True)
        loss, seg_l, recon_l = loss_fn(seg_logits, labels, recon, images)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item(), seg=seg_l.item(), recon=recon_l.item())
    
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch', get_not_nans=True)
    all_hd95 = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = sliding_window_inference(images, (128,128,128), 1, model, overlap=0.5)
            preds = torch.sigmoid(logits) > 0.5
            
            pred_list = [p for p in preds]
            label_list = [l for l in labels]
            dice_metric(y_pred=pred_list, y=label_list)
            
            for i in range(preds.shape[0]):
                for j in range(preds.shape[1]):
                    p = preds[i, j].cpu().numpy()
                    t = labels[i, j].cpu().numpy()
                    
                    if t.sum() == 0:
                        if p.sum() == 0:
                            hd95 = 0.0
                        else:
                            hd95 = 50.0
                    elif p.sum() == 0:
                        hd95 = 50.0
                    else:
                        try:
                            hd95 = binary.hd95(p, t, voxelspacing=None)
                            if np.isnan(hd95) or np.isinf(hd95):
                                hd95 = 50.0
                        except Exception:
                            hd95 = 50.0
                    
                    all_hd95.append(hd95)
    
    dice = dice_metric.aggregate()[0].cpu().numpy()
    mean_hd95 = np.mean(all_hd95) if all_hd95 else 50.0
    return dice, mean_hd95


def main():
    set_determinism(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Data
    train_dataset = BratsDataset(BRATS_TRAIN_FOLDERS, training=True, normalisation="zscore")
    val_dataset = BratsDataset(VALI_PATH, training=False, normalisation="zscore")
    
    if len(train_dataset) == 0:
        print("ERROR: No valid training patients found!")
        return
    
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Model
    model = CMADS_Net_MemoryEfficient(in_channels=4, out_channels=3, base_filters=16).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Optimizer & scheduler
    optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=100)
    loss_fn = CombinedLoss(dice_weight=1.0, recon_weight=0.05)
    
    # ========== FIXED: Use writable directories ==========
    # For resuming: load from input (read-only) but save to working (writable)
    INPUT_CHECKPOINT = "/kaggle/input/datasets/shahinhossain7611/dpca-models/latest_checkpoint.pth.tar"
    LATEST_CHECKPOINT = "/kaggle/working/latest_checkpoint.pth.tar"
    BEST_CHECKPOINT = "/kaggle/working/best_cmads_model.pth"
    BEST_FULL_CHECKPOINT = "/kaggle/working/best_checkpoint.pth.tar"
    
    # Try to resume from input checkpoint first, then fallback to working
    if os.path.isfile(INPUT_CHECKPOINT):
        print(f"Found checkpoint in input directory, loading...")
        start_epoch, best_dice = load_checkpoint(model, optimizer, scheduler, INPUT_CHECKPOINT, device)
        # Copy to working directory for future saves
        import shutil
        shutil.copy(INPUT_CHECKPOINT, LATEST_CHECKPOINT)
        print(f"Copied checkpoint to working directory: {LATEST_CHECKPOINT}")
    elif os.path.isfile(LATEST_CHECKPOINT):
        start_epoch, best_dice = load_checkpoint(model, optimizer, scheduler, LATEST_CHECKPOINT, device)
    else:
        print("No checkpoint found, starting from scratch.")
        start_epoch, best_dice = 1, 0.0
    
    # Training loop
    for epoch in range(start_epoch, 101):
        try:
            train_loss = train_epoch(model, train_loader, optimizer, loss_fn, epoch, 100, device)
            scheduler.step()
            
            # Save latest checkpoint to writable directory
            save_latest_checkpoint(model, optimizer, scheduler, epoch, best_dice, train_loss, LATEST_CHECKPOINT)
            
            # if epoch % 5 == 0 or epoch == 100:
            if epoch % 1 == 0 or epoch == 100:
                dice_scores, hd95 = validate(model, val_loader, device)
                avg_dice = np.mean(dice_scores)
                print(f"Epoch {epoch:3d} | Loss: {train_loss:.4f} | Dice ET/TC/WT: {dice_scores[0]:.4f}/{dice_scores[1]:.4f}/{dice_scores[2]:.4f} (Avg: {avg_dice:.4f}) | HD95: {hd95:.2f}")
                
                if avg_dice > best_dice:
                    best_dice = avg_dice
                    # Save best model to writable directory
                    torch.save(model.state_dict(), BEST_CHECKPOINT)
                    save_checkpoint({
                        'epoch': epoch,
                        'best_dice': best_dice,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                    }, BEST_FULL_CHECKPOINT)
                    print(f"*** New best: {best_dice:.4f} ***")
        
        except KeyboardInterrupt:
            print("\n⚠️ Training interrupted by user. Saving checkpoint before exit...")
            save_latest_checkpoint(model, optimizer, scheduler, epoch, best_dice, train_loss, LATEST_CHECKPOINT)
            print("Checkpoint saved. You can resume later by running the script again.")
            return
    
    print(f"\nTraining complete. Best Dice: {best_dice:.4f}")

if __name__ == "__main__":
    main()

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-17 06:23:27.108790: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776407007.337520     304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776407007.401640     304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776407007.925970     304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776407007.926016     304 computation_placer.cc:1

Using device: cuda
Validating 1041 patients...
Found 1041 valid patients, 0 invalid
Validating 210 patients...
Found 210 valid patients, 0 invalid
Train samples: 1041, Val samples: 210
Parameters: 1,228,071
Found checkpoint in input directory, loading...
Loading checkpoint '/kaggle/input/datasets/shahinhossain7611/dpca-models/latest_checkpoint.pth.tar'
Loaded checkpoint (epoch 28, best Dice 0.7903)
Copied checkpoint to working directory: /kaggle/working/latest_checkpoint.pth.tar


Epoch 29/100: 100%|██████████| 1041/1041 [25:48<00:00,  1.49s/it, loss=0.21, recon=2.94, seg=0.0631] 

Checkpoint saved to /kaggle/working/latest_checkpoint.pth.tar



/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:225: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:361: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out[idx_zm] += p


...

Checkpoint saved to /kaggle/working/latest_checkpoint.pth.tar
Epoch  34 | Loss: 0.2573 | Dice ET/TC/WT: 0.7779/0.6807/0.8334 (Avg: 0.7640) | HD95: 12.05


Epoch 35/100: 100%|██████████| 1041/1041 [25:54<00:00,  1.49s/it, loss=0.207, recon=1.65, seg=0.124] 

Checkpoint saved to /kaggle/working/latest_checkpoint.pth.tar


Epoch  35 | Loss: 0.2610 | Dice ET/TC/WT: 0.7720/0.7526/0.8840 (Avg: 0.8029) | HD95: 11.10
Checkpoint saved to /kaggle/working/best_checkpoint.pth.tar
*** New best: 0.8029 ***


Epoch 36/100:  94%|█████████▍| 980/1041 [24:23<01:30,  1.49s/it, loss=0.496, recon=2.02, seg=0.395] 

## Inference

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
import SimpleITK as sitk
from torch.utils.data import Dataset, DataLoader
from monai.inferers import sliding_window_inference
import random

# ======================== CONFIGURATION ========================

VALI_PATH = "/kaggle/input/datasets/shahajalalhossain/brats2021-training-validation-data/BraTS_2021_ValidationData"
MODEL_PATH = "/kaggle/working/best_cmads_model.pth"
OUTPUT_DIR = "/kaggle/working/segmentation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ======================== MODEL DEFINITIONS (MUST COME FIRST) ========================

class EfficientCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)
        self.to_out = nn.Linear(dim, dim)
        
    def forward(self, a, b):
        B, C, D, H, W = a.shape
        a_seq = a.flatten(2).transpose(1, 2)
        b_seq = b.flatten(2).transpose(1, 2)
        
        q = self.to_q(a_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        k = self.to_k(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        v = self.to_v(b_seq).view(B, -1, self.num_heads, C//self.num_heads).transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, -1, C)
        out = self.to_out(out)
        out = a_seq + out
        out = out.transpose(1, 2).view(B, C, D, H, W)
        return out


class LightweightDenoiser(nn.Module):
    def __init__(self, in_channels=4, base_filters=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
            nn.InstanceNorm3d(base_filters*4),
            nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*4, base_filters*2, 3, padding=1),
            nn.InstanceNorm3d(base_filters*2),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(base_filters*2, base_filters, 3, padding=1),
            nn.InstanceNorm3d(base_filters),
            nn.GELU(),
            nn.Conv3d(base_filters, in_channels, 3, padding=1),
        )
        
    def forward(self, x):
        features = self.encoder(x)
        recon = self.decoder(features)
        return torch.sigmoid(recon), features


class CMADS_Net_MemoryEfficient(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_filters=16):
        super().__init__()
        
        def make_encoder():
            return nn.ModuleList([
                nn.Sequential(
                    nn.Conv3d(in_channels, base_filters, 3, padding=1),
                    nn.InstanceNorm3d(base_filters),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters, base_filters*2, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*2),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*2, base_filters*4, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*4),
                    nn.LeakyReLU(),
                ),
                nn.Sequential(
                    nn.Conv3d(base_filters*4, base_filters*8, 3, stride=2, padding=1),
                    nn.InstanceNorm3d(base_filters*8),
                    nn.LeakyReLU(),
                )
            ])
        
        self.encoder_clean = make_encoder()
        self.encoder_noisy = make_encoder()
        self.cross_attn = EfficientCrossAttention(base_filters*8, num_heads=4)
        self.denoiser = LightweightDenoiser(in_channels, base_filters)
        
        self.decoder = nn.ModuleList([
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*8 + base_filters*4, base_filters*4, 3, padding=1),
                nn.InstanceNorm3d(base_filters*4),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*4 + base_filters*2, base_filters*2, 3, padding=1),
                nn.InstanceNorm3d(base_filters*2),
                nn.LeakyReLU(),
            ),
            nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
                nn.Conv3d(base_filters*2 + base_filters, base_filters, 3, padding=1),
                nn.InstanceNorm3d(base_filters),
                nn.LeakyReLU(),
            ),
            nn.Conv3d(base_filters, out_channels, 1)
        ])
        
    def forward(self, x_clean, noise_std=0.0, return_recon=False):
        if self.training and noise_std > 0:
            noise = torch.randn_like(x_clean) * noise_std
            x_noisy = x_clean + noise
        else:
            x_noisy = x_clean
        
        recon, _ = self.denoiser(x_noisy)
        
        clean_feats = []
        y = x_clean
        for block in self.encoder_clean:
            y = block(y)
            clean_feats.append(y)
        
        noisy_feats = []
        z = x_noisy
        for block in self.encoder_noisy:
            z = block(z)
            noisy_feats.append(z)
        
        fused_bottleneck = self.cross_attn(clean_feats[3], noisy_feats[3])
        
        recon_error_full = torch.abs(recon - x_clean).mean(dim=1, keepdim=True)
        
        error_map_size1 = F.interpolate(recon_error_full, size=clean_feats[1].shape[2:], mode='trilinear', align_corners=False)
        error_map_size2 = F.interpolate(recon_error_full, size=clean_feats[2].shape[2:], mode='trilinear', align_corners=False)
        
        calibrated_skip2 = clean_feats[2] * (1 + error_map_size2)
        calibrated_skip1 = clean_feats[1] * (1 + error_map_size1)
        calibrated_skip0 = clean_feats[0] * (1 + recon_error_full)
        
        d3 = self.decoder[0][0](fused_bottleneck)
        d3 = torch.cat([d3, calibrated_skip2], dim=1)
        d3 = self.decoder[0][1:](d3)
        
        d2 = self.decoder[1][0](d3)
        d2 = torch.cat([d2, calibrated_skip1], dim=1)
        d2 = self.decoder[1][1:](d2)
        
        d1 = self.decoder[2][0](d2)
        d1 = torch.cat([d1, calibrated_skip0], dim=1)
        d1 = self.decoder[2][1:](d1)
        
        logits = self.decoder[3](d1)
        
        if return_recon:
            return logits, recon
        return logits


# ======================== LOAD MODEL ========================

# Initialize model (same architecture as training)
model = CMADS_Net_MemoryEfficient(in_channels=4, out_channels=3, base_filters=16).to(device)

# Load trained weights
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint)
model.eval()
print(f"Model loaded from {MODEL_PATH}")


# ======================== DATASET CLASS FOR INFERENCE ========================

class BratsInferenceDataset(Dataset):
    """Simplified dataset for inference only (no augmentation)."""
    def __init__(self, patients_dir, normalisation="zscore"):
        self.patients_dir = patients_dir
        self.patients = sorted([x for x in Path(patients_dir).iterdir() if x.is_dir()])
        self.normalisation = normalisation
        
        self.modality_names = ['t1', 't1ce', 't2', 'flair']
        self.modality_suffixes = ['_t1.nii', '_t1ce.nii', '_t2.nii', '_flair.nii']
        self.seg_suffix = '_seg.nii'
        
        # Filter valid patients
        self.valid_patients = []
        for patient_dir in self.patients:
            if self._check_patient(patient_dir):
                self.valid_patients.append(patient_dir)
        
        print(f"Found {len(self.valid_patients)} valid patients out of {len(self.patients)}")
    
    def _check_patient(self, patient_dir):
        patient_id = patient_dir.name
        for suffix in self.modality_suffixes:
            if not (patient_dir / f"{patient_id}{suffix}").exists():
                return False
        if not (patient_dir / f"{patient_id}{self.seg_suffix}").exists():
            return False
        return True
    
    def load_nii(self, path):
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    
    def _zscore_normalise(self, img):
        img = img.astype(np.float32)
        nonzero = img[img > 0]
        if len(nonzero) > 0:
            mean, std = nonzero.mean(), nonzero.std()
            if std > 0:
                img = (img - mean) / std
        return img
    
    def __len__(self):
        return len(self.valid_patients)
    
    def __getitem__(self, idx):
        patient_dir = self.valid_patients[idx]
        patient_id = patient_dir.name
        
        # Load modalities
        patient_image = {}
        for mod_name, suffix in zip(self.modality_names, self.modality_suffixes):
            path = patient_dir / f"{patient_id}{suffix}"
            img = self.load_nii(path)
            patient_image[mod_name] = self._zscore_normalise(img)
        
        # Load segmentation
        seg_path = patient_dir / f"{patient_id}{self.seg_suffix}"
        seg = self.load_nii(seg_path)
        
        # Convert to ET, TC, WT
        et = (seg == 4).astype(np.float32)
        tc = np.logical_or(seg == 4, seg == 1).astype(np.float32)
        wt = np.logical_or(tc, seg == 2).astype(np.float32)
        label = np.stack([et, tc, wt], axis=0)
        
        # Stack modalities
        image = np.stack([patient_image['t1'], patient_image['t1ce'], 
                          patient_image['t2'], patient_image['flair']], axis=0)
        
        # Crop to non-zero region
        z_idx, y_idx, x_idx = np.nonzero(np.sum(image, axis=0) != 0)
        if len(z_idx) > 0:
            zmin, ymin, xmin = [max(0, int(np.min(arr)) - 1) for arr in (z_idx, y_idx, x_idx)]
            zmax, ymax, xmax = [int(np.max(arr)) + 1 for arr in (z_idx, y_idx, x_idx)]
            image = image[:, zmin:zmax, ymin:ymax, xmin:xmax]
            label = label[:, zmin:zmax, ymin:ymax, xmin:xmax]
        
        # Resize to 128x128x128
        image = self._resize_volume(image, (128, 128, 128))
        label = self._resize_volume(label, (128, 128, 128), is_label=True)
        
        return {
            'patient_id': patient_id,
            'image': torch.from_numpy(image.astype(np.float32)),
            'label': torch.from_numpy(label.astype(np.float32)),
        }
    
    def _resize_volume(self, volume, target_size, is_label=False):
        import torch.nn.functional as F_torch
        tensor = torch.from_numpy(volume).unsqueeze(0)
        mode = 'nearest' if is_label else 'trilinear'
        align = False if not is_label else None
        resized = F_torch.interpolate(tensor, size=target_size, mode=mode, align_corners=align)
        return resized.squeeze(0).numpy()


# ======================== INFERENCE FUNCTION ========================

def predict_patient(model, image, device):
    """Run inference on a single patient."""
    image = image.unsqueeze(0).to(device)  # Add batch dimension
    
    with torch.no_grad():
        # Sliding window inference
        logits = sliding_window_inference(
            inputs=image,
            roi_size=(128, 128, 128),
            sw_batch_size=1,
            predictor=model,
            overlap=0.5
        )
        
        # Convert to probabilities and binary masks
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
    
    return preds.squeeze(0).cpu().numpy(), probs.squeeze(0).cpu().numpy()


# ======================== VISUALIZATION FUNCTIONS ========================

def convert_to_3class_mask(et_mask, tc_mask, wt_mask):
    """
    Convert ET, TC, WT binary masks to a single 3-class mask for visualization.
    0: Background
    1: ET (Enhancing Tumor)
    2: TC (Tumor Core, excluding ET)
    3: WT (Whole Tumor, excluding TC)
    """
    combined = np.zeros(et_mask.shape, dtype=np.uint8)
    combined[wt_mask == 1] = 3  # WT (light green)
    combined[tc_mask == 1] = 2  # TC (light blue)
    combined[et_mask == 1] = 1  # ET (orange)
    return combined


def plot_segmentation_comparison(patient_id, image, gt_label, pred_label, slice_idx=None, save_path=None):
    """
    Plot comparison of ground truth vs prediction.
    """
    if slice_idx is None:
        slice_idx = image.shape[2] // 2
    
    # Get modalities
    t1 = image[0, :, :, :]
    t1ce = image[1, :, :, :]
    t2 = image[2, :, :, :]
    flair = image[3, :, :, :]
    
    # Convert masks
    gt_combined = convert_to_3class_mask(gt_label[0], gt_label[1], gt_label[2])
    pred_combined = convert_to_3class_mask(pred_label[0], pred_label[1], pred_label[2])
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 12))
    fig.suptitle(f'Patient: {patient_id} - Segmentation Comparison (Slice {slice_idx})', fontsize=16)
    
    modalities = [t1, t1ce, t2, flair]
    titles = ['T1', 'T1ce', 'T2', 'FLAIR']
    
    for col, (mod, title) in enumerate(zip(modalities, titles)):
        # Ground truth row
        axes[0, col].imshow(mod[:, :, slice_idx], cmap='gray', vmin=0, vmax=1)
        axes[0, col].imshow(gt_combined[:, :, slice_idx], cmap='jet', alpha=0.5, vmin=0, vmax=3)
        axes[0, col].axis('off')
        axes[0, col].set_title(f'{title} + GT')
        
        # Prediction row
        axes[1, col].imshow(mod[:, :, slice_idx], cmap='gray', vmin=0, vmax=1)
        axes[1, col].imshow(pred_combined[:, :, slice_idx], cmap='jet', alpha=0.5, vmin=0, vmax=3)
        axes[1, col].axis('off')
        axes[1, col].set_title(f'{title} + Prediction')
    
    # Legend
    legend_elements = [
        plt.Rectangle((0,0), 1, 1, facecolor='orange', alpha=0.6, label='ET (Enhancing Tumor)'),
        plt.Rectangle((0,0), 1, 1, facecolor='lightblue', alpha=0.5, label='TC (Tumor Core)'),
        plt.Rectangle((0,0), 1, 1, facecolor='lightgreen', alpha=0.4, label='WT (Whole Tumor)')
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()


def plot_multi_slice_comparison(patient_id, image, gt_label, pred_label, num_slices=5, save_dir=None):
    """
    Plot multiple axial slices for better visualization.
    """
    depth = image.shape[2]
    slice_indices = np.linspace(depth//4, 3*depth//4, num_slices, dtype=int)
    
    fig, axes = plt.subplots(num_slices, 3, figsize=(15, 4*num_slices))
    fig.suptitle(f'Patient: {patient_id} - Multi-Slice Segmentation Comparison', fontsize=16)
    
    gt_combined = convert_to_3class_mask(gt_label[0], gt_label[1], gt_label[2])
    pred_combined = convert_to_3class_mask(pred_label[0], pred_label[1], pred_label[2])
    flair = image[3, :, :, :]
    
    for i, slice_idx in enumerate(slice_indices):
        # Ground Truth
        axes[i, 0].imshow(flair[:, :, slice_idx], cmap='gray', vmin=0, vmax=1)
        axes[i, 0].imshow(gt_combined[:, :, slice_idx], cmap='jet', alpha=0.5, vmin=0, vmax=3)
        axes[i, 0].axis('off')
        axes[i, 0].set_title(f'Slice {slice_idx} - GT')
        
        # Prediction
        axes[i, 1].imshow(flair[:, :, slice_idx], cmap='gray', vmin=0, vmax=1)
        axes[i, 1].imshow(pred_combined[:, :, slice_idx], cmap='jet', alpha=0.5, vmin=0, vmax=3)
        axes[i, 1].axis('off')
        axes[i, 1].set_title(f'Slice {slice_idx} - Prediction')
        
        # Difference (red = FP, blue = FN)
        gt_mask = gt_combined[:, :, slice_idx] > 0
        pred_mask = pred_combined[:, :, slice_idx] > 0
        diff = np.zeros((*gt_mask.shape, 3))
        fp = np.logical_and(pred_mask, ~gt_mask)
        fn = np.logical_and(~pred_mask, gt_mask)
        diff[fp, 0] = 1  # Red for false positives
        diff[fn, 2] = 1  # Blue for false negatives
        
        axes[i, 2].imshow(flair[:, :, slice_idx], cmap='gray', vmin=0, vmax=1)
        axes[i, 2].imshow(diff, alpha=0.6)
        axes[i, 2].axis('off')
        axes[i, 2].set_title(f'Slice {slice_idx} - Difference (Red=FP, Blue=FN)')
    
    plt.tight_layout()
    
    if save_dir:
        save_path = os.path.join(save_dir, f"{patient_id}_multi_slice.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()


def compute_metrics(gt_label, pred_label):
    """Compute Dice score for each class."""
    dice_scores = {}
    class_names = ['ET', 'TC', 'WT']
    
    for i, name in enumerate(class_names):
        gt = gt_label[i].flatten()
        pred = pred_label[i].flatten()
        
        intersection = np.sum(gt * pred)
        union = np.sum(gt) + np.sum(pred)
        
        if union == 0:
            dice = 1.0 if intersection == 0 else 0.0
        else:
            dice = (2.0 * intersection) / union
        
        dice_scores[name] = dice
    
    return dice_scores


def save_nifti_prediction(patient_id, pred_label, output_dir):
    """Save prediction as NIfTI file."""
    final_mask = np.zeros(pred_label.shape[1:], dtype=np.uint8)
    final_mask[pred_label[0] == 1] = 4  # ET
    final_mask[pred_label[1] == 1] = 1  # TC
    final_mask[pred_label[2] == 1] = 2  # WT
    
    img = sitk.GetImageFromArray(final_mask)
    sitk.WriteImage(img, os.path.join(output_dir, f"{patient_id}_pred_seg.nii"))
    print(f"Saved prediction to {output_dir}/{patient_id}_pred_seg.nii")


def visualize_specific_patient(patient_id):
    """Visualize a specific patient by ID."""
    dataset = BratsInferenceDataset(VALI_PATH)
    
    # Find patient index
    patient_idx = None
    for idx, patient in enumerate(dataset.valid_patients):
        if patient.name == patient_id:
            patient_idx = idx
            break
    
    if patient_idx is None:
        print(f"Patient {patient_id} not found!")
        print(f"Available patients (first 10): {[p.name for p in dataset.valid_patients[:10]]}")
        return
    
    patient_data = dataset[patient_idx]
    patient_id = patient_data['patient_id']
    image = patient_data['image']
    gt_label = patient_data['label'].numpy()
    
    print(f"\n{'='*60}")
    print(f"Processing patient: {patient_id}")
    print(f"{'='*60}")
    
    # Run inference
    pred_label, probs = predict_patient(model, image, device)
    
    # Compute metrics
    dice_scores = compute_metrics(gt_label, pred_label)
    print(f"Dice Scores - ET: {dice_scores['ET']:.4f}, TC: {dice_scores['TC']:.4f}, WT: {dice_scores['WT']:.4f}")
    print(f"Mean Dice: {np.mean(list(dice_scores.values())):.4f}")
    
    # Create output directory for this patient
    patient_dir = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(patient_dir, exist_ok=True)
    
    # Visualize
    middle_slice = image.shape[2] // 2
    plot_segmentation_comparison(
        patient_id, 
        image.numpy(), 
        gt_label, 
        pred_label,
        slice_idx=middle_slice,
        save_path=os.path.join(patient_dir, "comparison_mid_slice.png")
    )
    
    plot_multi_slice_comparison(
        patient_id,
        image.numpy(),
        gt_label,
        pred_label,
        num_slices=5,
        save_dir=patient_dir
    )
    
    save_nifti_prediction(patient_id, pred_label, patient_dir)
    
    return pred_label, dice_scores


# ======================== RUN INFERENCE ========================

if __name__ == "__main__":
    # Visualize a specific patient
    # Try a patient that exists in your validation set
    # Based on your earlier output, valid patients exist like 'BraTS2021_01457'
    
    # Get list of available patients first
    temp_dataset = BratsInferenceDataset(VALI_PATH)
    if len(temp_dataset.valid_patients) > 0:
        print(f"\nAvailable patients: {[p.name for p in temp_dataset.valid_patients[:5]]}")
        
        # Visualize the first valid patient
        first_patient = temp_dataset.valid_patients[0].name
        print(f"\nVisualizing first patient: {first_patient}")
        visualize_specific_patient(first_patient)
    else:
        print("No valid patients found in validation set!")
    
    print(f"\nAll results saved to: {OUTPUT_DIR}")

In [ ]:
# !rm -rf /kaggle/working/segmentation_results